In [2]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-06-02 10:45:49--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt.1’

input.txt.1         100%[===================>]   1.06M  1.22MB/s    in 0.9s    

2026-06-02 10:45:50 (1.22 MB/s) - ‘input.txt.1’ saved [1115394/1115394]



In [3]:
with open("input.txt", 'r', encoding="utf-8") as f:
    text = f.read()

In [4]:
print(f"Length of the dataset character {len(text)}")

Length of the dataset character 1115394


In [5]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [6]:
chars = sorted(list(set(text)))
val = ([(chr(ord(char))) for char in chars])
vocab_size = len(chars)
print(type(val), ''.join(val))
print(vocab_size)


<class 'list'> 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


## Character Level Language Model Tokenizer

#### For the encoder I am creating a map, every unique character in the vocab gets an ID, and the map contains the <char : ID> sequence
#### For the decoder I am passing a list of ineterges through another map that contains the <ID : char> sequence

In [7]:
stoi = { ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])
print(encode("hello"))
print(decode(encode("hello")))

[46, 43, 50, 50, 53]
hello


In [8]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
## encode using our own encoder sequence from <char : ID> and torch size is the same as of the entire length of the text
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [9]:
n = int(0.9 * len(data))
# starting from 90% i.e 1003854 till the end we have the test data and starting from 0 to the 90% is the train data
train_data = data[:n]
val_data = data[n:]

In [10]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [11]:
x = train_data[:block_size]
print(x, len(x))
y = train_data[1:block_size+1]
print(y, len(y))

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"For the context of {context} the target is {target}")


tensor([18, 47, 56, 57, 58,  1, 15, 47]) 8
tensor([47, 56, 57, 58,  1, 15, 47, 58]) 8
For the context of tensor([18]) the target is 47
For the context of tensor([18, 47]) the target is 56
For the context of tensor([18, 47, 56]) the target is 57
For the context of tensor([18, 47, 56, 57]) the target is 58
For the context of tensor([18, 47, 56, 57, 58]) the target is 1
For the context of tensor([18, 47, 56, 57, 58,  1]) the target is 15
For the context of tensor([18, 47, 56, 57, 58,  1, 15]) the target is 47
For the context of tensor([18, 47, 56, 57, 58,  1, 15, 47]) the target is 58


In [12]:
start_idx = [1078327,  453969,   41646,  671252]
for d in start_idx:
    for i in range(block_size):
        data_idx = data[d+i].item()
        dt = decode([data_idx])
        print(''.join(dt))
    print("="*10)



u
n
k
i
n
d
 
b
o
p
e
f
u
l
 
l
O
L
U
M
N
I
A
:
i
c
e
 
o
v
e
r


In [13]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):
    data = train_data if split == "train" else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    ## returns me 4 numbers basically offset values from the entire list of numbers (dataset)
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    # we will get 8 characters * 4 repetitions since the batch size is 4 and the block size is 8
    return x,y,ix

xb , yb, ix = get_batch("split")
print("inputs: ")
print(xb.shape, xb)

for i in xb:
    dt_item = [j.item() for j in i]
    chr = ''.join([decode([dt_item_i]) for dt_item_i in dt_item])
    print(i, chr)
print("="*10)
print("targets: ")
print(yb.shape, yb)

for i in yb:
    dt_item = [j.item() for j in i]
    chr = ''.join([decode([dt_item_i]) for dt_item_i in dt_item])
    print(i, chr)
print("-"*6)
# print(f"The slicing for context is {xb[0,:1]}")
# print(f"The slicing for the target is {yb[0,0]}")


for b in range(batch_size): # 0-4 times
    print(f"Starting to sample from the {ix[b]}th offset of the dataset ")
    for t in range(block_size): 
        context = xb[b, :t+1]
        target = yb[b,t]
        print(f"When context is {context.tolist()} : {decode(data[context.tolist()].tolist())} then the target is {target}")
    print("----")
  

inputs: 
torch.Size([4, 8]) tensor([[ 6,  1, 52, 53, 58,  1, 58, 47],
        [ 6,  1, 54, 50, 39, 52, 58, 43],
        [ 1, 58, 46, 47, 57,  1, 50, 47],
        [ 0, 32, 46, 43, 56, 43,  1, 42]])
tensor([ 6,  1, 52, 53, 58,  1, 58, 47]) , not ti
tensor([ 6,  1, 54, 50, 39, 52, 58, 43]) , plante
tensor([ 1, 58, 46, 47, 57,  1, 50, 47])  this li
tensor([ 0, 32, 46, 43, 56, 43,  1, 42]) 
There d
targets: 
torch.Size([4, 8]) tensor([[ 1, 52, 53, 58,  1, 58, 47, 50],
        [ 1, 54, 50, 39, 52, 58, 43, 58],
        [58, 46, 47, 57,  1, 50, 47, 60],
        [32, 46, 43, 56, 43,  1, 42, 53]])
tensor([ 1, 52, 53, 58,  1, 58, 47, 50])  not til
tensor([ 1, 54, 50, 39, 52, 58, 43, 58])  plantet
tensor([58, 46, 47, 57,  1, 50, 47, 60]) this liv
tensor([32, 46, 43, 56, 43,  1, 42, 53]) There do
------
Starting to sample from the 29535th offset of the dataset 
When context is [6] : C then the target is 1
When context is [6, 1] : Ci then the target is 52
When context is [6, 1, 52] : Cie then the ta

In [14]:
print(xb)
# this is the input to the transformer, a batch of 4 parallel and size of 8

tensor([[ 6,  1, 52, 53, 58,  1, 58, 47],
        [ 6,  1, 54, 50, 39, 52, 58, 43],
        [ 1, 58, 46, 47, 57,  1, 50, 47],
        [ 0, 32, 46, 43, 56, 43,  1, 42]])


### Using bi gram language model n = 2 

In [15]:
import torch
import torch.nn as nn 
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size): # vocab size is 65 for us 
        super().__init__()
        # each token directly reads the logits of the next token! 
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    def forward(self, idx, targets=None): 
        # idx and targets are both (B,T) tensor of integers
        logits = self.token_embedding_table(idx) # (B,T,C) Batch time channel tensor
        # in pytorch we use a B C T struct for the log loss, so we need to reshape our tensor
        # the loss should be somewhere close to -ln(1/65) which is around ~ 4.17 but we are getting 4.87 so some short diffusion has occured somewhere down the line.
        if targets is None: 
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss
    
    def generate(self, idx, max_new_tokens):
        # idx is (B,T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim = 1 )
        return idx
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb,yb)
print(logits.shape, loss)
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))

torch.Size([32, 65]) tensor(4.4913, grad_fn=<NllLossBackward0>)

SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


In [16]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)
batch_size = 32 # for production
for steps in range(10000):
    xb,yb,ix = get_batch("train")
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.382369041442871


In [17]:
print(decode(m.generate(idx=torch.zeros((1,1), dtype=torch.long), max_new_tokens=100)[0].tolist()))


lso br. ave aviasurf my, yxMPZI ivee iuedrd whar ksth y h bora s be hese, woweee; the! KI 'de, ulsee


### This is the point upto the biagram model but yet the tokens are not talking to each other, up next the tokens shall talk to each other

In [28]:
torch.manual_seed(1337)
B,T,C = 4,8,2
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [ ]:
# Using for loop
xbow = torch.zeros((B,T,C))
for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)

In [ ]:
# Usinf the matrix multiplication method
wei = torch.tril(torch.ones(T,T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei  @ x # B,T,T @ B,T,C


tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]]) tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [37]:
# Using the softmax layer
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0 ,float('-inf'))
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x